In [ ]:
# @title 1. Clean Environment & Install Dependencies
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Work from a stable root (Colab: /content)
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(ROOT)

# Make sure we don't accumulate nested clones when re-running the notebook
repo_dir = ROOT / "motion-diffusion-model"
if repo_dir.exists() and not (repo_dir / ".git").exists():
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    print("📂 Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/GuyTevet/motion-diffusion-model.git", str(repo_dir)], check=True)
else:
    print("✅ Repository already present. Using existing clone.")

os.chdir(repo_dir)

print("📦 Installing Python dependencies (quiet mode)...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "scikit-image", "imageio"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio==2.33.0", "scikit-image==0.22.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smplx", "chumpy", "trimesh", "moviepy", "gradio", "gdown"], check=True)
subprocess.run(["apt-get", "update"], check=False)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)

print("✅ Environment ready.")


In [ ]:
# @title 2. Prepare Minimal HumanML3D Text-Only Dataset
import os
import urllib.request
from pathlib import Path

DATA_ROOT = Path("dataset/HumanML3D")
TEXT_DIR = DATA_ROOT / "texts"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

# Download mean/std statistics (small files) from a mirror
mirror_base = "https://huggingface.co/OpenRobotLab/MotionMillion/resolve/main/mean_std"
for fname in ["Mean.npy", "Std.npy"]:
    target = DATA_ROOT / fname
    if not target.exists():
        print(f"⬇️ Downloading {fname} ...")
        urllib.request.urlretrieve(f"{mirror_base}/{fname}", target)

# Build a tiny text-only split so the loader is non-empty without full motion data
splits = {
    "train": ["colab_sample_0001", "colab_sample_0002"],
    "val": ["colab_sample_0001"],
    "test": ["colab_sample_0002"],
}

def write_split_file(name, ids):
    with open(DATA_ROOT / f"{name}.txt", "w") as f:
        for item in ids:
            f.write(f"{item}
")

for split_name, id_list in splits.items():
    write_split_file(split_name, id_list)

# Minimal caption files (format: caption#tokens#f_tag#to_tag)
text_payloads = {
    "colab_sample_0001": "a person is waving#person waving#0#0
",
    "colab_sample_0002": "a person is turning around#person turning#0#0
",
}
for name, content in text_payloads.items():
    with open(TEXT_DIR / f"{name}.txt", "w") as f:
        f.write(content)

print("✅ Text-only dataset prepared in", DATA_ROOT)


In [ ]:
# @title 3. Download Pretrained Model
import os
import zipfile
import subprocess
from pathlib import Path

file_id = "1PE0PK8e5a5j-7-Xhs5YET5U5pGh0c821"
zip_filename = Path("humanml_trans_enc_512.zip")
model_dir = Path("save")
final_model_path = model_dir / "humanml_trans_enc_512" / "model000200000.pt"
model_dir.mkdir(parents=True, exist_ok=True)

if not final_model_path.exists():
    print("⬇️ Downloading pretrained weights (Google Drive)...")
    subprocess.run(["gdown", "--id", file_id, "-O", str(zip_filename)], check=True)
    print("📂 Unzipping model archive...")
    with zipfile.ZipFile(zip_filename, "r") as zf:
        zf.extractall(model_dir)
    zip_filename.unlink(missing_ok=True)
else:
    print("✅ Model already downloaded.")

print("Model location:", final_model_path)


In [ ]:
# @title 4. Sanity Check: Dataset Loader
from pathlib import Path

split_file = Path("dataset/HumanML3D/test.txt")
print("Test split contains", sum(1 for _ in open(split_file)), "entries")
print("Mean present:", (Path("dataset/HumanML3D/Mean.npy").exists()))
print("Std present:", (Path("dataset/HumanML3D/Std.npy").exists()))


In [ ]:
# @title 5. Run a Quick Generation (no Gradio)
import glob
import subprocess
import sys
from pathlib import Path

model_path = "save/humanml_trans_enc_512/model000200000.pt"
text_prompt = "a person is waving their hands"
out_dir = Path("save/humanml_trans_enc_512")

if not Path(model_path).exists():
    raise FileNotFoundError("Model weights missing. Please run step 3.")

cmd = [
    sys.executable, "-m", "sample.generate",
    "--model_path", model_path,
    "--text_prompt", text_prompt,
    "--motion_length", "2.0",
    "--num_repetitions", "1",
    "--batch_size", "1",
    "--device", "cpu",
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:
", result.stderr)

latest = sorted(glob.glob(str(out_dir / "samples_*/sample*_rep*.mp4")))
if latest:
    print("Latest sample:", latest[-1])
else:
    print("No sample video found. Check logs above for details.")
